In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import IsolationForest

from sklearn.svm import OneClassSVM

from sklearn.neighbors import LocalOutlierFactor

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
df = pd.read_csv(
    "behavior_feature_engineered_data.csv"
)

In [3]:
df.head()

,user_id,country,device_type,browser,failed_attempts,login_status,is_anomaly,hour,day_of_week,is_weekend,max_normal_failed_attempts,usual_start_hour,usual_end_hour,is_new_country,is_new_device,is_new_browser,unusual_login_hour,excessive_failed_attempts,behavior_risk_score,high_risk_login
0,0,4,4,3,1,1,0,13,6,1,16,0,23,0,0,0,0,0,0,0
1,0,4,4,3,3,1,0,20,6,1,16,0,23,0,0,0,0,0,0,0
2,0,4,4,3,1,1,0,15,0,0,16,0,23,0,0,0,0,0,0,0
3,0,4,4,3,1,1,0,21,0,0,16,0,23,0,0,0,0,0,0,0
4,0,4,4,3,0,1,0,21,0,0,16,0,23,0,0,0,0,0,0,0


In [4]:
X = df.drop(columns=["is_anomaly"])

y = df["is_anomaly"]

In [7]:
X.select_dtypes(include="str").columns

Index([], dtype='str')

In [8]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [10]:
iso_model = IsolationForest(

    contamination=0.05,

    random_state=42
)

iso_model.fit(X_train_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [11]:
iso_preds = iso_model.predict(X_test_scaled)

In [12]:
iso_preds = np.where(
    iso_preds == -1,
    1,
    0
)

In [13]:
print("Isolation Forest Results\n")

print(confusion_matrix(y_test, iso_preds))

print("\n")

print(classification_report(y_test, iso_preds))

Isolation Forest Results

[[1900    0]
 [   0  100]]


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1900
           1       1.00      1.00      1.00       100

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [14]:
svm_model = OneClassSVM(

    kernel='rbf',

    gamma='scale',

    nu=0.05
)

svm_model.fit(X_train_scaled)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.05
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [15]:
svm_preds = svm_model.predict(X_test_scaled)

svm_preds = np.where(
    svm_preds == -1,
    1,
    0
)

In [16]:
print("One-Class SVM Results\n")

print(confusion_matrix(y_test, svm_preds))

print("\n")

print(classification_report(y_test, svm_preds))

One-Class SVM Results

[[1846   54]
 [  65   35]]


              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1900
           1       0.39      0.35      0.37       100

    accuracy                           0.94      2000
   macro avg       0.68      0.66      0.67      2000
weighted avg       0.94      0.94      0.94      2000



In [17]:
lof_model = LocalOutlierFactor(

    contamination=0.05,

    novelty=True
)

lof_model.fit(X_train_scaled)

,"n_neighbors n_neighbors: int, default=20Number of neighbors to use by default for :meth:`kneighbors` queries.If n_neighbors is larger than the number of samples provided,all samples will be used.",20
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf is size passed to :class:`BallTree` or :class:`KDTree`. This canaffect the speed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float, default=2Parameter for the Minkowski metric from:func:`sklearn.metrics.pairwise_distances`. When p = 1, thisis equivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. When fitting this is used to define thethreshold on the scores of the samples.- if 'auto', the threshold is determined as in the original paper,- if a float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"novelty novelty: bool, default=FalseBy default, LocalOutlierFactor is only meant to be used for outlierdetection (novelty=False). Set novelty to True if you want to useLocalOutlierFactor for novelty detection. In this case be aware thatyou should only use predict, decision_function and score_sampleson new unseen data and not on the training set; and note that theresults obtained this way may differ from the standard LOF results... versionadded:: 0.20",True
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [18]:
lof_preds = lof_model.predict(X_test_scaled)

lof_preds = np.where(
    lof_preds == -1,
    1,
    0
)

In [19]:
print("Local Outlier Factor Results\n")

print(confusion_matrix(y_test, lof_preds))

print("\n")

print(classification_report(y_test, lof_preds))

Local Outlier Factor Results

[[1797  103]
 [  93    7]]


              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1900
           1       0.06      0.07      0.07       100

    accuracy                           0.90      2000
   macro avg       0.51      0.51      0.51      2000
weighted avg       0.91      0.90      0.90      2000



In [20]:
models = {

    "Isolation Forest": iso_preds,

    "One-Class SVM": svm_preds,

    "Local Outlier Factor": lof_preds
}

for name, preds in models.items():

    print(f"\n{name}")

    print("-" * 30)

    print(
        "Accuracy:",
        accuracy_score(y_test, preds)
    )

    print(
        "Precision:",
        precision_score(y_test, preds)
    )

    print(
        "Recall:",
        recall_score(y_test, preds)
    )

    print(
        "F1 Score:",
        f1_score(y_test, preds)
    )


Isolation Forest
------------------------------
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

One-Class SVM
------------------------------
Accuracy: 0.9405
Precision: 0.39325842696629215
Recall: 0.35
F1 Score: 0.37037037037037035

Local Outlier Factor
------------------------------
Accuracy: 0.902
Precision: 0.06363636363636363
Recall: 0.07
F1 Score: 0.06666666666666667
